# Quantifaya AutoDirector — CogVideoX Colab GPU Worker

This notebook runs the **automated GPU generation worker** for AutoDirector. It listens on Redis for video-generation jobs enqueued by the API (`ColabProvider`), generates each clip with **CogVideoX-2b** on the free Colab T4 GPU, uploads the mp4 to **Backblaze B2**, and pushes the result back to Redis.

## Setup once
1. **Runtime → Change runtime type → T4 GPU** (free tier).
2. Step **Cells 3–4** ask for your Hugging Face token and connection credentials.
3. Run **Cell 9** (the worker loop) and leave it running.

The worker stays alive and processes jobs as the API enqueues them. Stop it with the square stop button when done.

In [ ]:
# 1. Install dependencies (run once)
!pip install -q torch diffusers transformers accelerate sentencepiece \
             protobuf redis boto3 imageio[ffmpeg]

In [ ]:
# 2. Imports
import json, os, time, uuid, getpass
import torch
import redis
import boto3
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

In [ ]:
# 3. Hugging Face auth (model is gated — token required)
hf_token = os.environ.get('HUGGINGFACE_TOKEN') or getpass.getpass('Hugging Face token: ')
if not hf_token:
    raise ValueError('A Hugging Face token is required to download CogVideoX-2b.')
print('HF token set:', bool(hf_token))

In [ ]:
# 4. Connection credentials (paste your values or set env vars)
B2_KEY_ID        = os.environ.get('B2_KEY_ID', '')
B2_APPLICATION_KEY = os.environ.get('B2_APPLICATION_KEY', '')
B2_BUCKET        = os.environ.get('B2_BUCKET_NAME', 'quantifaya')
B2_ENDPOINT      = os.environ.get('B2_ENDPOINT_URL', 'https://s3.us-east-005.backblazeb2.com')
PUBLIC_BASE      = os.environ.get('B2_PUBLIC_URL_BASE', f'https://f005.backblazeb2.com/file/{B2_BUCKET}')
REDIS_URL        = os.environ.get('REDIS_URL', '')

assert B2_KEY_ID and B2_APPLICATION_KEY, 'Set B2_KEY_ID and B2_APPLICATION_KEY'
assert REDIS_URL, 'Set REDIS_URL (Upstash or local)'
print('Credentials OK.')

In [ ]:
# 5. Load the model (T4-friendly: CogVideoX-2b, fp16, CPU offload)
MODEL_ID = 'THUDM/CogVideoX-2b'
print('Loading', MODEL_ID, '...')
pipe = CogVideoXPipeline.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, token=hf_token
)
pipe.enable_model_cpu_offload()
print('Model loaded.')

In [ ]:
# 6. B2 upload helper
s3 = boto3.client(
    's3',
    endpoint_url=B2_ENDPOINT,
    aws_access_key_id=B2_KEY_ID,
    aws_secret_access_key=B2_APPLICATION_KEY,
    region_name='us-east-005',
)

def upload_to_b2(local_path: str, key: str) -> str:
    s3.upload_file(local_path, B2_BUCKET, key, ExtraArgs={'ContentType': 'video/mp4'})
    return f'{PUBLIC_BASE}/{key}'

In [ ]:
# 7. Worker loop — processes jobs from colab:videojobs
r = redis.from_url(REDIS_URL, decode_responses=True)

def process_job(payload: dict):
    job_id = payload.get('job_id', uuid.uuid4().hex[:8])
    result_key = payload.get('result_key', f'colab:results:{job_id}')
    output_key = payload.get('output_key') or f"episodes/{payload.get('episode_id','unknown')}/intro.mp4"
    duration = int(payload.get('duration_secs', 8))

    num_frames = max(16, min(49, int(duration * 8)))
    local_path = f'/content/{job_id}.mp4'

    try:
        frames = pipe(
            prompt=payload.get('prompt', ''),
            negative_prompt=payload.get('negative_prompt', ''),
            num_frames=num_frames,
            num_inference_steps=50,
            guidance_scale=6.0,
        ).frames[0]
        export_to_video(frames, local_path, fps=8)
        public_url = upload_to_b2(local_path, output_key)
        r.lpush(result_key, json.dumps({'status': 'success', 'url': public_url}))
        print(f'[done] {job_id} -> {public_url}')
    except Exception as e:
        r.lpush(result_key, json.dumps({'status': 'failed', 'error': str(e)}))
        print(f'[failed] {job_id}: {e}')

print('Worker listening on colab:videojobs... (Ctrl-C / stop button to exit)')
while True:
    try:
        raw = r.brpop('colab:videojobs', timeout=0)
        if raw:
            process_job(json.loads(raw[1]))
    except KeyboardInterrupt:
        print('Worker stopped.')
        break
    except Exception as e:
        print(f'[error] {e}')
        time.sleep(5)

## Worker status
The loop is running. Monitor the `[done]` / `[failed]` lines above.

To stop the worker, press the **stop / square** button in the Colab toolbar. To restart, re-run Cell 7 (the worker loop). The model stays loaded in memory across restarts of the loop cell.